In [1]:
%%capture
%pip install black[jupyter] blackcellmagic

In [2]:
import IPython
import black

# from google.colab import output


def format_cell(result):
    shell = IPython.get_ipython()
    raw_cell = shell.history_manager.input_hist_raw[-1]

    try:
        formatted = black.format_str(raw_cell, mode=black.Mode())
        if formatted.strip() != raw_cell.strip():
            # We update the cell content for the next time it is viewed/edited
            shell.set_next_input(formatted, replace=True)
    except Exception as e:
        # Skip formatting if there's a syntax error during the black process
        pass


# Register the hook to run after every cell execution
get_ipython().events.register("post_run_cell", format_cell)
print("Auto-formatter active: Cells will reformat automatically upon execution.")

Auto-formatter active: Cells will reformat automatically upon execution.


In [3]:
%%capture
%pip install uaibot

In [6]:

import numpy as np
import uaibot as ub
import qpsolvers

from dataclasses import dataclass
from typing import cast
from uaibot.simobjects import SimObject


@dataclass(frozen=True)
class SimulationConfig:
    dt: float = 0.01
    tmax: float = 12.0
    k0: float = 1.0
    gamma: float = 500.0
    epsilon: float = 1e-3
    lambda_d: float = 0.01


@dataclass(frozen=True)
class CircularPathConfig:
    radius: float = 0.1
    x_center: float = 0.6
    y_center: float = 0.0
    z_center: float = 0.8
    omega: float = np.pi / 2


CONFIG = SimulationConfig()
PATH = CircularPathConfig()


def desired_position(p: float) -> np.ndarray:
    return np.array([
        [PATH.x_center],
        [PATH.y_center + PATH.radius * np.cos(p)],
        [PATH.z_center + PATH.radius * np.sin(p)],
    ])


def desired_velocity(p: float) -> np.ndarray:
    return np.array([
        [0.0],
        [-PATH.radius * np.sin(p)],
        [PATH.radius * np.cos(p)],
    ])


def create_path_point_cloud(num_points: int = 200) -> np.ndarray:
    points = [
        desired_position(2 * np.pi * i / (num_points - 1))
        for i in range(num_points)
    ]
    return np.column_stack(points)


def task_function(r: np.ndarray) -> np.ndarray:
    K = 0.25
    r_norm = np.linalg.norm(r)
    return -K * np.sqrt(r_norm) * np.sign(r)


def compute_task_error(
    s_e: np.ndarray,
    z_e: np.ndarray,
    p: float,
    z_d: np.ndarray,
) -> np.ndarray:
    r = np.zeros((4, 1))
    r[0:3] = s_e - desired_position(p)
    r[3, 0] = 1.0 - (z_d.T @ z_e)[0, 0]
    return r


def build_task_jacobian(
    Jg: np.ndarray,
    z_e: np.ndarray,
    z_d: np.ndarray,
) -> np.ndarray:
    n = Jg.shape[1]
    Jr = np.zeros((4, n))

    Jr[0:3, :] = Jg[0:3, :]
    Jr[3, :] = (
        z_d.T
        @ ub.Utils.S(np.matrix(z_e))
        @ Jg[3:6, :]
    )

    return Jr


def solve_control_qp(
    P_qp: np.ndarray,
    q_dot_0: np.ndarray,
    Jr: np.ndarray,
    task_rhs: np.ndarray,
    q_current: np.ndarray,
    q_min: np.ndarray,
    q_max: np.ndarray,
    dt: float,
    epsilon: float,
) -> np.ndarray:

    n = q_current.shape[0]
    m = Jr.shape[0]

    q_qp = np.concatenate([
        -CONFIG.lambda_d * q_dot_0.flatten(),
        np.zeros(m),
    ])

    A_qp = np.hstack([Jr, -np.identity(m)])

    lb_dq = ((q_min - q_current) / dt + epsilon).flatten()
    ub_dq = ((q_max - q_current) / dt - epsilon).flatten()

    lb_qp = np.concatenate([lb_dq, np.full(m, -np.inf)])
    ub_qp = np.concatenate([ub_dq, np.full(m, np.inf)])

    solution = qpsolvers.solve_qp(
        P=P_qp,
        q=q_qp,
        A=A_qp,
        b=task_rhs.flatten(),
        lb=lb_qp,
        ub=ub_qp,
        solver="osqp",
    )

    if solution is None:
        raise RuntimeError("QP solver returned no solution.")

    return solution[:n].reshape((n, 1))


robot = ub.Robot.create_kuka_lbr_iiwa()

n = len(robot.q)
m = 4

q_current = np.asarray(robot.q, dtype=float).reshape((n, 1))

q_limits = np.asarray(robot._joint_limit, dtype=float)
q_min = q_limits[:, 0].reshape((n, 1))
q_max = q_limits[:, 1].reshape((n, 1))

q_mean = (q_max + q_min) / 2.0
q_range_sq = np.square(q_max - q_min)

z_d = np.array([[1.0], [0.0], [0.0]])

point_cloud = create_path_point_cloud()

target_pos_pc = ub.PointCloud(
    points=np.matrix(point_cloud),
    size=0.03,
    color="purple",
)

ball_tr = ub.Ball(
    htm=np.asmatrix(np.identity(4)),
    radius=0.02,
    color="cyan",
)

sim = ub.Simulation(
    cast(list[SimObject], [robot, target_pos_pc, ball_tr])
)

P_qp = np.block([
    [CONFIG.lambda_d * np.identity(n), np.zeros((n, m))],
    [np.zeros((m, n)), np.identity(m)],
])

p = 0.0
t = 0.0

for _ in range(round(CONFIG.tmax / CONFIG.dt)):

    Jg, fk = robot.jac_geo(q_current)

    Jg = np.asarray(Jg, dtype=float)
    fk = np.asarray(fk, dtype=float)

    z_e = fk[0:3, 2].reshape((3, 1))
    s_e = fk[0:3, 3].reshape((3, 1))

    desired_pos = desired_position(p)
    desired_vel = desired_velocity(p)

    r = compute_task_error(
        s_e=s_e,
        z_e=z_e,
        p=p,
        z_d=z_d,
    )

    r_norm_sq = (r.T @ r)[0, 0]

    p_dot = PATH.omega / (
        1.0 + CONFIG.gamma * r_norm_sq
    )

    ff = np.vstack([
        -desired_vel * p_dot,
        np.zeros((1, 1)),
    ])

    Jr = build_task_jacobian(
        Jg=Jg,
        z_e=z_e,
        z_d=z_d,
    )

    grad_H = (q_current - q_mean) / q_range_sq
    q_dot_0 = -CONFIG.k0 * grad_H

    task_rhs = task_function(r) - ff

    try:
        u = solve_control_qp(
            P_qp=P_qp,
            q_dot_0=q_dot_0,
            Jr=Jr,
            task_rhs=task_rhs,
            q_current=q_current,
            q_min=q_min,
            q_max=q_max,
            dt=CONFIG.dt,
            epsilon=CONFIG.epsilon,
        )

    except Exception as exc:
        print(f"Optimization failure at t={t:.2f}: {exc}")
        break

    q_current = q_current + u * CONFIG.dt
    p = p + p_dot * CONFIG.dt
    t = t + CONFIG.dt

    robot.add_ani_frame(
        time=t,
        q=np.matrix(q_current),
    )

    ball_tr.add_ani_frame(
        time=t,
        htm=ub.Utils.trn(np.matrix(desired_pos)),
    )

sim.run()
